In [ ]:
# Author: M. Riley Owens (GitHub: mrileyowens)

# This file makes cutout images from DJA of the JADES F775W dropout galaxies and 
# then merges those images with the figures of the corresponding SED fits

In [ ]:
import sys

import os
import glob

from pypdf import PdfWriter

from astropy.io import fits

sys.path.append(os.path.abspath('..'))

from mrileyowens.dja import cutout

In [ ]:
# Set common directories
home = os.getcwd()
figs = f'{home}/figs'
results = f'{home}/results'

def cutouts():

    '''
    Make photometric cutouts of the JADES F775W dropout galaxy catalog
    '''

    # Open the JADES F775W dropout galaxy catalog
    hdul = fits.open(f'{results}/catalogs/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog_f775w_dropouts_final.fits')

    # Get the IDs and coordinates from the catalog
    ids, ra, dec = hdul[1].data['ID'], hdul[1].data['RA'], hdul[1].data['DEC']

    # For each object in the catalog
    for i, _ in enumerate(ra):

        # Get a cutout of the object from DJA in the filters used to construct the selection
        cutout(ra[i], dec[i], filters=['f435w', 'f606w', 'f775w','f814w','f850lp','f090w-clear','f115w-clear','f150w-clear','f200w-clear','f277w-clear','f335m-clear','f356w-clear','f410m-clear','f444w-clear'], 
            id=ids[i], cutouts_in_row=5, save=True, dir=f'{figs}/cutouts')

def merge():

    '''
    Merge the DJA photometric cutout PDFs with the corresponding JADES SED fits PDFs
    '''

    # For each cutout PDF
    for file in glob.glob(f'{figs}/cutouts/*.pdf'):

        # Make an empty PDF to merge the individual PDFs into
        merger = PdfWriter()

        # For the PDFs of the DJA photometric cutouts and JADES SED fits
        for pdf in [file, f'{figs}/bagpipes_fits/{os.path.basename(file)}']:

            # Try to append the PDF to the merged PDF
            try:
                merger.append(pdf)
            except FileNotFoundError:
                pass

        # Save the merged PDF
        merger.write(f'{figs}/cutouts_bagpipes_fits/{os.path.basename(file)}')

In [ ]:
cutouts()

In [ ]:
merge()